# YouTube 자막 → Markdown(.md) 변환기

유튜브 URL을 입력하면 **영상은 다운로드하지 않고 자막만** 가져와 Markdown 파일로 저장합니다.

- 업로더가 제공한 자막과 유튜브 자동 생성 자막을 모두 검색합니다.
- 한국어/영어 등 원하는 언어 코드를 지정할 수 있습니다.
- 자막의 타임스탬프를 Markdown에 포함할 수 있습니다.
- 원문 보존을 위해 자막 문장을 임의로 요약하거나 중복 제거하지 않습니다.

> 참고: 영상에 자막이 없거나 유튜브/영상 소유자의 제한으로 자막을 가져올 수 없는 경우에는 실패할 수 있습니다. 본 노트북은 자막 추출 도구이며 음성 자체를 STT로 변환하지는 않습니다.


## 1. 필요한 패키지 설치

처음 한 번 실행하면 됩니다. 설치 후 오류가 나면 커널을 재시작한 뒤 다시 실행하세요.

In [ ]:
# %pip -q install -U yt-dlp


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. 다운로드 설정

- `VIDEO_URL`: 유튜브 영상 주소
- `LANGUAGES`: 우선 가져올 자막 언어. 예: `["ko", "en"]`
- `INCLUDE_AUTO_CAPTIONS`: 자동 생성 자막도 검색할지 여부
- `INCLUDE_TIMESTAMPS`: Markdown에 각 자막 구간의 시작 시간을 표시할지 여부

`ko`는 한국어, `en`은 영어입니다. 다른 언어는 ISO 언어 코드로 입력할 수 있습니다.

In [208]:
from pathlib import Path

VIDEO_URL = "https://www.youtube.com/watch?v=FNfQ6IQQjao&t=70s"  # 여기에 유튜브 URL 입력
LANGUAGES = ["ko"]
INCLUDE_AUTO_CAPTIONS = True
INCLUDE_TIMESTAMPS = True

# 결과 저장 폴더
OUTPUT_DIR = Path.cwd() / "youtube_transcripts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "영상ID" in VIDEO_URL or not VIDEO_URL.startswith(("https://", "http://")):
    raise ValueError("VIDEO_URL에 실제 유튜브 영상 주소를 입력해주세요.")

print("저장 폴더:", OUTPUT_DIR.resolve())

저장 폴더: /Users/heojiyeon/Desktop/밀라논나/youtube_transcripts


## 3. 자막 다운로드

아래 셀은 영상 파일은 받지 않고 자막 파일만 다운로드합니다. 사용 가능한 자막을 확인한 뒤, 지정한 언어의 자막을 가져옵니다.

In [209]:
import yt_dlp
from pathlib import Path

download_options = {
    "skip_download": True,
    "writesubtitles": True,
    "writeautomaticsub": INCLUDE_AUTO_CAPTIONS,
    "subtitleslangs": LANGUAGES,
    "subtitlesformat": "vtt",
    "outtmpl": str(OUTPUT_DIR / "%(title).150B_%(id)s.%(ext)s"),
    "quiet": False,
    "no_warnings": False,
    "ignoreerrors": False,
}

with yt_dlp.YoutubeDL({"quiet": True, "no_warnings": True}) as ydl:
    info = ydl.extract_info(VIDEO_URL, download=False)
    video_title = info.get("title", "youtube_video")
    video_id = info.get("id", "unknown")
    print("영상 제목:", video_title)
    print("영상 ID:", video_id)

with yt_dlp.YoutubeDL(download_options) as ydl:
    ydl.download([VIDEO_URL])

vtt_files = sorted(OUTPUT_DIR.glob(f"*{video_id}*.vtt"))

if not vtt_files:
    # 파일명 템플릿/플랫폼에 따라 ID 검색이 안 될 경우 최근 VTT 파일을 확인
    vtt_files = sorted(OUTPUT_DIR.glob("*.vtt"), key=lambda p: p.stat().st_mtime, reverse=True)

if not vtt_files:
    raise FileNotFoundError(
        "VTT 자막 파일을 찾지 못했습니다. 해당 영상에 자막이 있는지, "
        "LANGUAGES 설정이 맞는지 확인해주세요."
    )

print("\n다운로드된 자막 파일:")
for path in vtt_files:
    print("-", path.name)

영상 제목: [라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC241030방송
영상 ID: FNfQ6IQQjao
[youtube] Extracting URL: https://www.youtube.com/watch?v=FNfQ6IQQjao&t=70s
[youtube] FNfQ6IQQjao: Downloading webpage


[youtube] FNfQ6IQQjao: Downloading visionos player API JSON
[youtube] FNfQ6IQQjao: Downloading m3u8 information
[info] FNfQ6IQQjao: Downloading subtitles: ko
[info] FNfQ6IQQjao: Downloading 1 format(s): 137+251
Deleting existing file /Users/heojiyeon/Desktop/밀라논나/youtube_transcripts/[라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC2410_FNfQ6IQQjao.ko.vtt
[info] Writing video subtitles to: /Users/heojiyeon/Desktop/밀라논나/youtube_transcripts/[라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC2410_FNfQ6IQQjao.ko.vtt


[download] Destination: /Users/heojiyeon/Desktop/밀라논나/youtube_transcripts/[라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC2410_FNfQ6IQQjao.ko.vtt
[download] 100% of  135.58KiB in 00:00:00 at 772.72KiB/s

다운로드된 자막 파일:
- [라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC2410_FNfQ6IQQjao.ko.vtt


## 4. VTT 자막을 Markdown으로 변환

자막의 HTML 태그와 VTT 형식 제어 정보만 정리합니다. 문장 자체는 임의로 합치거나 중복 제거하지 않습니다. 타임스탬프 포함 여부는 위 설정을 따릅니다.

In [210]:
import re
import html
from datetime import timedelta

TIMESTAMP_RE = re.compile(
    r"^(?:(\d{2,}):)?(\d{2}):(\d{2})[.,](\d{3})\s*-->\s*"
)

def timestamp_to_seconds(timestamp: str) -> float:
    """VTT 타임스탬프를 초 단위로 변환합니다."""
    timestamp = timestamp.replace(",", ".")
    parts = timestamp.split(":")
    if len(parts) == 3:
        hours, minutes, seconds = parts
    else:
        hours = "0"
        minutes, seconds = parts
    return int(hours) * 3600 + int(minutes) * 60 + float(seconds)

def format_timestamp(seconds: float) -> str:
    total = int(seconds)
    hours = total // 3600
    minutes = (total % 3600) // 60
    secs = total % 60
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"

def parse_vtt(vtt_text: str):
    """VTT cue를 (시작 시간, 자막 텍스트) 목록으로 읽습니다."""
    lines = vtt_text.replace("\ufeff", "").splitlines()
    cues = []
    current_start = None
    text_lines = []
    skipping_block = False

    def flush():
        nonlocal current_start, text_lines
        if current_start is not None:
            raw = " ".join(part.strip() for part in text_lines if part.strip())
            # HTML/VTT 태그 제거 후 HTML 엔티티 복원
            clean = re.sub(r"<[^>]*>", "", raw)
            clean = html.unescape(clean).strip()
            if clean:
                cues.append((current_start, clean))
        current_start = None
        text_lines = []

    for line in lines:
        stripped = line.strip()

        if stripped.startswith(("NOTE", "STYLE", "REGION")):
            flush()
            skipping_block = True
            continue

        if not stripped:
            flush()
            skipping_block = False
            continue

        if skipping_block:
            continue

        match = TIMESTAMP_RE.match(stripped)
        if match:
            flush()
            current_start = timestamp_to_seconds(stripped.split("-->")[0].strip())
            continue

        # cue 식별자나 WEBVTT 헤더 등은 건너뜁니다.
        if stripped == "WEBVTT" or stripped.startswith("Kind:") or stripped.startswith("Language:"):
            continue

        if current_start is not None:
            text_lines.append(stripped)

    flush()
    return cues

def vtt_to_markdown(vtt_path: Path, include_timestamps: bool = True) -> Path:
    text = vtt_path.read_text(encoding="utf-8-sig", errors="replace")
    cues = parse_vtt(text)

    if not cues:
        raise ValueError(f"자막 내용을 읽지 못했습니다: {vtt_path.name}")

    md_lines = [
        f"# {video_title}",
        "",
        f"- Video URL: {VIDEO_URL}",
        f"- Subtitle file: `{vtt_path.name}`",
        f"- Subtitle segments: {len(cues):,}",
        "",
        "---",
        "",
    ]

    for start, caption in cues:
        if include_timestamps:
            md_lines.append(f"**[{format_timestamp(start)}]** {caption}")
        else:
            md_lines.append(caption)
        md_lines.append("")

    md_path = vtt_path.with_suffix(".md")
    md_path.write_text("\n".join(md_lines), encoding="utf-8")
    return md_path

md_files = []
for vtt_file in vtt_files:
    try:
        md_file = vtt_to_markdown(vtt_file, INCLUDE_TIMESTAMPS)
        md_files.append(md_file)
        print(f"변환 완료: {md_file.name}")
    except Exception as exc:
        print(f"변환 실패 ({vtt_file.name}): {exc}")

if not md_files:
    raise RuntimeError("Markdown으로 변환된 파일이 없습니다.")

변환 완료: [라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC2410_FNfQ6IQQjao.ko.md


## 5. 결과 확인 및 파일 경로

생성된 Markdown 파일의 크기와 앞/뒤 일부 내용을 확인합니다. 자막이 끝까지 들어왔는지 영상과 대조해 확인하는 것이 좋습니다.

In [211]:
for md_file in md_files:
    content = md_file.read_text(encoding="utf-8")
    print("=" * 80)
    print("파일:", md_file.resolve())
    print("크기:", f"{md_file.stat().st_size:,}", "bytes")
    print("\n[처음 1,000자]")
    print(content[:1000])
    print("\n[마지막 500자]")
    print(content[-500:])

파일: /Users/heojiyeon/Desktop/밀라논나/youtube_transcripts/[라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC2410_FNfQ6IQQjao.ko.md
크기: 65,795 bytes

[처음 1,000자]
# [라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC241030방송

- Video URL: https://www.youtube.com/watch?v=FNfQ6IQQjao&t=70s
- Subtitle file: `[라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC2410_FNfQ6IQQjao.ko.vtt`
- Subtitle segments: 838

---

**[00:03]** 같은 클래스메이트는 이제 돌체 나름

**[00:03]** 같은 클래스메이트는 이제 돌체 나름 브랜드 저 거기 조고 좋아합니다 예

**[00:08]** 브랜드 저 거기 조고 좋아합니다 예

**[00:08]** 브랜드 저 거기 조고 좋아합니다 예 짝 클라스메이트

**[00:10]** 짝 클라스메이트

**[00:10]** 짝 클라스메이트 같이

**[00:12]** 같이

**[00:12]** 같이 체 아님은 너는 동양인이라 이렇게

**[00:15]** 체 아님은 너는 동양인이라 이렇게

**[00:15]** 체 아님은 너는 동양인이라 이렇게 손이

**[00:17]** 손이

**[00:17]** 손이 섬세하다고네 그 라인은 굉장히 그

**[00:19]** 섬세하다고네 그 라인은 굉장히 그

**[00:19]** 섬세하다고네 그 라인은 굉장히 그 디테일에 강하다 그러면 안 보는

**[00:21]** 디테일에 강하다 그러면 안 보는

**[00:21]** 디테일에 강하다 그러면 안 보는 척하고 와서 그렇게 보고 가고

**[00:24

## 6. 다운로드 링크 표시 (Jupyter 환경)

Jupyter Notebook/Lab에서 실행하면 아래 링크를 눌러 Markdown 파일을 열거나 저장할 수 있습니다.

In [212]:
from IPython.display import display, Markdown

for md_file in md_files:
    relative_path = md_file.relative_to(Path.cwd()) if md_file.is_relative_to(Path.cwd()) else md_file
    link = relative_path.as_posix()
    display(Markdown(f"- [Markdown 파일 다운로드: {md_file.name}]({link})"))

- [Markdown 파일 다운로드: [라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC2410_FNfQ6IQQjao.ko.md](youtube_transcripts/[라디오스타] '사람 자체가 명품이 되는 법'을 알려주는 94만 유튜버 밀라논나의 인생관ㅣ#밀라논나 #장도연 MBC2410_FNfQ6IQQjao.ko.md)